In [ ]:
import matplotlib
from datascience import *
%matplotlib inline
import matplotlib.pyplot as plots
import numpy as np
plots.style.use('fivethirtyeight')

In [ ]:
# Some functions for plotting. You don't have to understand how any of the functions in this cell work,
# since they use things we haven't learned about in DSCI 100.

def resize_window(lim=3.5):
    plots.xlim(-lim, lim)
    plots.ylim(-lim, lim)
    
def draw_line(slope=0, intercept=0, x=make_array(-4, 4), color='#1e90ff'):
    y = x*slope + intercept
    plots.plot(x, y, color=color, lw=3)
    
def draw_vertical_line(x_position, color='black'):
    x = make_array(x_position, x_position)
    y = make_array(-4, 4)
    plots.plot(x, y, color=color, lw=3)
    
def make_correlated_data(r):
    "Make up data for analysis"
    x = np.random.normal(0, 1, 1000)
    z = np.random.normal(0, 1, 1000)
    y = r*x + (np.sqrt(1-r**2))*z
    return x, y

def r_scatter(r):
    """Generate a scatter plot with a correlation approximately r"""
    plots.figure(figsize=(5,5))
    x, y = make_correlated_data(r)
    plots.scatter(x, y, color='darkblue', s=20)
    plots.xlim(-4, 4)
    plots.ylim(-4, 4)
    
def r_table(r):
    """
    Generate a table of 1000 data points with a correlation approximately r
    """
    np.random.seed(8)
    x, y = make_correlated_data(r)
    return Table().with_columns('x', x, 'y', y)

In [ ]:
# Functions from lecture 29
def standard_units(x):
    "Convert any array of numbers to standard units."
    return (x - np.average(x)) / np.std(x)

def correlation(t, x, y):
    """t is a table; x and y are column labels"""
    x_in_standard_units = standard_units(t.column(x))
    y_in_standard_units = standard_units(t.column(y))
    return np.average(x_in_standard_units * y_in_standard_units)

# Lecture 30 Linear Regression

## Interpreting Relationships
### Nonlinearity in Relationships

When looking at a scatter, to determine if there is a relationship, we will look for a pattern. If the data looks like  
it has a linear relationship, we can use the Correlation Coefficient, $r$, to quantify that relationship.

Some relationships are not linear so $r$ is not a helpful measurement in establishing the strength of that relationship.  

See the example below.

In [ ]:
new_x = np.arange(-4, 4.1, 0.5)
nonlinear = Table().with_columns(
        'x', new_x,
        'y', new_x**2
    )
nonlinear.scatter('x', 'y', s=30, color='r')

***QUESTION: Does this data appear to have a relationship?***

In [ ]:
# COMPLETE: Use the correlation(t, x, y) function on this table to determine r. 



Since the relationship is not linear, $r$ makes it appear as if there is no relationship.  
    
There is obviously a pattern so $r$ is not helpful.

### Outliers

Outliers are individual data points that lie way outside reasonable values. 

These can cause $r$ to show no relationship when there is one or a strong relationship when it's weak. 

See the example below.

In [ ]:
# Example scatter with a very linear relationship. 
line = Table().with_columns(
        'x', make_array(1, 2, 3, 4),
        'y', make_array(1, 2, 3, 4)
    )
line.scatter('x', 'y', s=30, color='r')

In [ ]:
# COMPLETE: Use the correlation(t, x, y) functionn on this table to determine r. 



In [ ]:
# How is r affected when we insert an outlier far off the trend of the graph.
outlier = Table().with_columns(
        'x', make_array(1, 2, 3, 4, 5),
        'y', make_array(1, 2, 3, 4, 0)
    )
outlier.scatter('x', 'y', s=30, color='r')

In [ ]:
# COMPLETE: Use the correlation(t, x, y) function on this table to determine r. 



Only one outlier can greatly affect $r$.

***QUESTION: What can cause outliers?***

### Ecological Correlations

Ecological correlation is a statistical relationship between two variables measured at the group level instead of the individual level. 

This can also have an interesting affect on $r$. 

See the example below. 

Load in the data for 2014 SAT Data for 51 regions: The 50 States and DC.   
Describe the table.
                                   
_Note: The scores are the averages across the state._

In [ ]:
sat2014 = Table.read_table('sat2014.csv').sort('State')
sat2014

In [ ]:
# COMPLETE: Plot the scatter for Reading and Math correlation.



***QUESTION: What would you estimate the correlation coefficient, $r$, to be?***

In [ ]:
# COMPLETE: Use the correlation(t, x, y) function to quantify the relationship between Critical Reading and Math. 



### Be Careful

***QUESTION: What does each point on the scatter plot represent?***



***QUESTION: Is this helpful in predicting an individuals expected score on the SAT? Why or Why Not?***



***QUESTION: Would $r$ be the same if we used individuals?***


Looking again at the 2014 SAT Data for the 51 U.S. regions, we want to compare the participation rates.

Each region has a different participation because of the requirements of that state regarding the SATs.

***QUESTION: What types of requirements might affect the participation rates in each state?***
    
***QUESTION: What will the function below return when applied to the participation rate column of the SAT Data?***

In [ ]:
def rate_code(x):
    if x <= 25:
        return 'low'
    elif x <= 75:
        return 'medium'
    else:
        return 'high'

In [ ]:
# COMPLETE: Run the rate code function on the Participation Rate column. 



In [ ]:
# COMPLETE: Create a new table adding the Rate Code column with the new rate codes. 

sat2014_rate = 
sat2014_rate

In [ ]:
# COMPLETE: View the same scatter with the rate code grouping. Show the scatter command specs to see how to group. 



***QUESTION: What is happening with the grouping?***

* **Low Participation:** ?
* **Medium Particpation:** ?
* **High Participation:** ?


In [ ]:
# COMPLETE: Display a table of only the regions with low rate code. Do not reassign. 



In [ ]:
# COMPLETE: How many regions had a low rate code?



***QUESTION: Do you find anything interesting about the states that have low participation rates?***

Ecological Correlation is when you compare the averages of a measurement for a group, instead of the measurements of each individual in those groups.  
This is not a true correlation so we can't use it to determine relationships at the individual level between the two variables.

## Using Correlation to Create a Prediction Model

The Correlation coefficient, $r$, can also help us identify the straight line that the points are clustered around.   
    
Using the nearest neighbors can help us predict an average value for each y-value given a particular x-value based on the line's equation.

We will create a table of ficticious data that has a correlation of a particular $r$ for several different values.  
The r_table(r) function imported earlier will create table of 1000 random values for $x$ and $y$ that has a relationship defined by $r$.


In [ ]:
# COMPLETE: Use the r_table(r) function to create a sample with r = 0.99.

example = 
example.show(3)

In [ ]:
#COMPLETE: Create a scatter of comparing x and y. 


resize_window()

In [ ]:
# Nearest neighbor prediction function will select an interval near a given x, 
# then take the average of the y values in that interval. 

def nn_prediction_example(x_val):
    """ Predicts y-value for x based on the example table. Nearest Neighbor """
    neighbors = example.where('x', are.between(x_val - .25, x_val + .25))
    return np.mean(neighbors.column('y'))   

In [ ]:
# COMPLETE: Use the function to predict the y-value from an x-value of -2.25.



In [ ]:
# COMPLETE: Add the column of the predicted y-values to the example table. 

example = 
example.show(3)

In [ ]:
# COMPLETE: Create a scatter that shows the y-values and the predicted y-values. 


resize_window()

In [ ]:
# Creates the same graph overlayed with a line of slope=1.
example.scatter('x')
draw_line(slope=1)
resize_window()

***QUESTION: What was the assigned $r$ value?***

***QUESTION: How well does the line with a slope of 1 align with the predicted values?***

***QUESTION: Why do you think that is?***


In [ ]:
# COMPLETE: Use the r_table(r) function to create a sample with r = 0 and create a scatter to display the results.

example = 


resize_window()

In [ ]:
# Creates a table that shows the predicted y-value for each x-value.
example = example.with_columns(
    'Predicted y', 
    example.apply(nn_prediction_example, 'x'))

In [ ]:
# Creates a scatter that displays the predicted y-values and a line with a slope of 0.

example.scatter('x')
draw_line(slope = 0)
resize_window()

In [ ]:
# COMPLETE: Use the r_table(r) function to create a sample with r = 0.5 and create a scatter to display the results.

example = 

resize_window()

In [ ]:
# Adds a prediction line that shows what we would predict if x was 1.5?
# It also adds a line with slope of 1.
example = r_table(0.5)
example.scatter('x', 'y')
resize_window()
draw_vertical_line(1.5)
draw_line(slope=1, intercept=0, color='red')

***QUESTION: Is the intersection of the two lines a good prediction for this data? Why or Why not?***

In [ ]:
# Adds the predicted values with the prediction line that shows what we would predict if x was 1.5?
# It also adds a line with slope of 1.
example = example.with_column('Predicted y', example.apply(nn_prediction_example, 'x'))
example.scatter('x')
draw_line(slope=1, color='red')
draw_vertical_line(1.5)
resize_window()

***QUESTION: Do you need to adjust your answer now that you can see the predicted values? Why or Why not?***

In [ ]:
# Adds a line with slope of 0.5 to the scatter and removes the prediction line. 

example.scatter('x')
draw_line(slope=1, intercept=0, color='red')
draw_line(slope=0.5, intercept=0)
resize_window()

In [ ]:
# Creates a scatter where r = 0.7, shows predicted values, line with slope = 1, and line with slope = 0.7.

example = r_table(0.7)
example = example.with_column('Predicted y', example.apply(nn_prediction_example, 'x'))
example.scatter('x')
draw_line(slope=1, intercept=0, color='red')
draw_line(slope=0.7, intercept=0, color='dodgerblue')
resize_window()

***QUESTION: What assumptions might you make based on the scatters we've seen?***

## Linear Regression: Defining the Model

Linear regression creates a model that defines the relationship between two variables.  
Since the relationship is linear the model uses the equation of the line. 

When $r$ is in standard units, the line that defines the model is $y=mx$</br>
Where $m$, the slope, is equal to $r$ and the y-intercept, $b$, is equal to zero.

We will use the standard_units(x) function to standardize/normalize our data and the correlation(t, x, y) to find $r$.

Recall: 
 * What is the mean of normalized data?
 * What is the standard deviation of normalized data?


When the data is not standardized, the model will follow the defintion of a line $y=mx+b$.</br>
Where $m$, the slope, is equal to $\dfrac{r*y_{SD}}{x_{SD}}$</br>
and $b$, the y-intercept, is equal to $y_{mean} - slope*x_{mean}$

In [ ]:
# Creates a function that determines the slope of data. 

def slope(t, x, y):
    """ Computes the slope of the regression line, like correlation above """
    r = correlation(t, x, y)
    y_sd = np.std(t.column(y))
    x_sd = np.std(t.column(x))
    return r * y_sd / x_sd


In [ ]:
# Creates a function that determines the y-intercept of data. 

def intercept(t, x, y):
    """ Computes the intercept of the regression line, like slope above """
    x_mean = np.mean(t.column(x))
    y_mean = np.mean(t.column(y))
    return y_mean - slope(t, x, y)*x_mean

In [ ]:
# Confirm that the slope function works on an example where r is defined as 0.5.
example = r_table(0.5)
slope(example, 'x', 'y')

## Heights Data and Regression Line


In [ ]:
# Note: Child heights are the **adult** heights of children in a family
families = Table.read_table('family_heights.csv')

parent_avgs = (families.column('father') + families.column('mother'))/2
heights = Table().with_columns(
    'Parent Average', parent_avgs,
    'Child', families.column('child'),
)
heights.show(5)

In [ ]:
def nn_prediction_height(p_avg):
    """Predict the height of a child whose parents have a parent average height of p_avg.
    
    The prediction is the average height of the children whose parent average height is
    in the range p_avg plus or minus 0.5.
    """
    
    close_points = heights.where('Parent Average', are.between(p_avg-0.5, p_avg + 0.5))
    return np.average(close_points.column('Child')) 

In [ ]:
# Adds a column of the predicted values.

heights_with_predictions = heights.with_column(
    'Nearest neighbor prediction', 
    heights.apply(nn_prediction_height, 'Parent Average'))
heights_with_predictions.show(5)

In [ ]:
# COMPLETE: Show a scatter of the Parent Averages.



In [ ]:
predicted_heights_slope = slope(heights, 'Parent Average', 'Child')
predicted_heights_intercept = intercept(heights, 'Parent Average', 'Child')
[predicted_heights_slope, predicted_heights_intercept]

***QUESTION: How would you write the information found in the format of a linear regression model? Hint use LaTeX.***

$y =$

***QUESTION: What is this calculating and what is the input?***

***QUESTION: Does the y-intercept make sense as an actual data point?***


In [ ]:
# COMPLETE: Use the model to create a Regression Prediction column. 
heights_with_predictions = 
    
    
)
heights_with_predictions

In [ ]:
# Show the scatter with the regression predictions and the nearest neighbor predications.

heights_with_predictions.scatter('Parent Average')


***QUESTION: What observations can you make?***

In [ ]:
# Create a table that shows the standardized values of the parents' average and child's height. 

Standard_Heights = Table().with_columns(
    'Parent Average',  standard_units(heights.column('Parent Average')), 
    'Child', standard_units(heights.column('Child'))
    )
Standard_Heights

In [ ]:
# Creates a scatter comparing the standardized values. 

Standard_Heights.scatter(0, 1)
plots.xlim(-3, 3)
plots.ylim(-3, 3);

***QUESTION: What do you notice when comparing the standardized scatter with the non-standardized scatter?***

In [ ]:
# Shows the slope and y-intercept of the standardized data. 

predicted_Sheights_slope = slope(Standard_Heights, 'Parent Average', 'Child')
predicted_Sheights_intercept = intercept(Standard_Heights, 'Parent Average', 'Child')
[predicted_Sheights_slope, predicted_Sheights_intercept]

***QUESTION: How would you write the information found in the format of a linear regression model?***

$y = $

In [ ]:
# Shows the correlation of the standardized data.

correlation(Standard_Heights, 'Parent Average', 'Child')

In [ ]:
Standard_Heights.scatter(0, 1)
draw_line(slope=predicted_Sheights_slope, intercept=predicted_Sheights_intercept, color='red')
plots.xlim(-3, 3)
plots.ylim(-3, 3);

***QUESTION: What do you notice when comparing the standardized regression line with the non-standardized regression line?***